# DDRI Station-Hour(대여소-시간 단위) Model Comparison(모델 비교)

대표 대여소 15개 `station-hour(대여소-시간 단위)` 데이터셋으로 기준선 모델(baseline)과 트리 기반 모델을 비교한다.

- Train(학습): `2023`
- Validation(검증): `2024`
- Final Train(최종 재학습): `2023 + 2024`
- Test(최종 평가): `2025`
- 비교 모델: `LightGBM`, `CatBoost`
- 학습 목표 함수(objective) 비교: `RMSE objective(제곱평균제곱근오차 기준 학습)`, `Poisson objective(Poisson 분포 기준 학습)`


In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import koreanize_matplotlib
plt.rcParams['font.family'] = 'Apple SD Gothic Neo'
plt.rcParams['axes.unicode_minus'] = False
import pandas as pd
import seaborn as sns
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
sns.set_theme(style='whitegrid', rc={'font.family': 'Apple SD Gothic Neo', 'axes.unicode_minus': False})

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [2]:
ROOT = Path('/Users/cheng80/Desktop/ddri_work')
RAW_DATA_DIR = ROOT / '3조 공유폴더/대표대여소_예측데이터_15개/raw_data'
OUTPUT_DATA_DIR = ROOT / 'works/05_prediction_long/output/data'
IMAGE_DIR = ROOT / 'works/05_prediction_long/output/images'
OUTPUT_DATA_DIR.mkdir(parents=True, exist_ok=True)
IMAGE_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = RAW_DATA_DIR / 'ddri_prediction_long_train_2023_2024.csv'
TEST_PATH = RAW_DATA_DIR / 'ddri_prediction_long_test_2025.csv'


## 1. 데이터 불러오기
대표 대여소 15개 `station-hour(대여소-시간 단위)` 데이터셋을 읽고 `datetime(일시)`을 재구성한다.


In [3]:
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

for df in [train, test]:
    df['date'] = pd.to_datetime(df['date'])
    df['datetime'] = pd.to_datetime(
        df['date'].dt.strftime('%Y-%m-%d') + ' ' + df['hour'].astype(str).str.zfill(2) + ':00:00'
    )

train.shape, test.shape

((263160, 16), (131400, 16))

## 2. 시계열 피처 생성

시간 단위 수요는 직전 1시간, 24시간, 1주 전 패턴의 영향을 많이 받을 수 있으므로 `lag`(과거 시점 값)와 `rolling`(이동 통계) 피처를 추가한다.

- `lag`: 직전 1시간, 24시간 전, 1주 전처럼 과거 같은 대상의 값을 가져온 피처
- `rolling`: 최근 일정 구간의 평균이나 표준편차를 계산한 피처

In [4]:
combined = pd.concat([train.assign(dataset='train'), test.assign(dataset='test')], ignore_index=True)
combined = combined.sort_values(['station_id', 'datetime']).reset_index(drop=True)

grouped_target = combined.groupby('station_id')['rental_count']
combined['lag_1h'] = grouped_target.shift(1)
combined['lag_24h'] = grouped_target.shift(24)
combined['lag_168h'] = grouped_target.shift(168)
combined['rolling_mean_24h'] = grouped_target.shift(1).rolling(24).mean()
combined['rolling_mean_168h'] = grouped_target.shift(1).rolling(168).mean()
combined['rolling_std_24h'] = grouped_target.shift(1).rolling(24).std()

train_model = combined[combined['dataset'] == 'train'].copy()
test_model = combined[combined['dataset'] == 'test'].copy()

FEATURE_COLUMNS = [
    'station_id', 'station_group', 'cluster', 'mapped_dong_code', 'hour', 'weekday', 'month', 'holiday',
    'temperature', 'humidity', 'precipitation', 'wind_speed',
    'lag_1h', 'lag_24h', 'lag_168h', 'rolling_mean_24h', 'rolling_mean_168h', 'rolling_std_24h'
]
CATEGORICAL_COLUMNS = ['station_id', 'station_group', 'cluster', 'mapped_dong_code', 'hour', 'weekday', 'month', 'holiday']

train_model[FEATURE_COLUMNS].head(3)

,station_id,station_group,cluster,mapped_dong_code,hour,weekday,month,holiday,temperature,humidity,precipitation,wind_speed,lag_1h,lag_24h,lag_168h,rolling_mean_24h,rolling_mean_168h,rolling_std_24h
0,2312,주거 도착형,2,11680565,0,6,1,1,-2.0,81,0.0,9.3,NaN,NaN,NaN,NaN,NaN,NaN
1,2312,주거 도착형,2,11680565,1,6,1,1,-1.7,82,0.0,9.3,0.0,NaN,NaN,NaN,NaN,NaN
2,2312,주거 도착형,2,11680565,2,6,1,1,-0.8,77,0.0,10.8,0.0,NaN,NaN,NaN,NaN,NaN


## 3. 학습/검증/평가 분리
`2023`을 학습(train), `2024`를 검증(validation), `2025`를 최종 평가(test)로 둔다.


In [5]:
train_2023 = train_model[train_model['date'] < '2024-01-01'].copy()
valid_2024 = train_model[train_model['date'] >= '2024-01-01'].copy()
full_train = train_model.copy()

X_train = train_2023[FEATURE_COLUMNS].copy()
y_train = train_2023['rental_count'].copy()

X_valid = valid_2024[FEATURE_COLUMNS].copy()
y_valid = valid_2024['rental_count'].copy()

X_full = full_train[FEATURE_COLUMNS].copy()
y_full = full_train['rental_count'].copy()

X_test = test_model[FEATURE_COLUMNS].copy()
y_test = test_model['rental_count'].copy()

for frame in [X_train, X_valid, X_full, X_test]:
    for column in CATEGORICAL_COLUMNS:
        frame[column] = frame[column].astype('category')

train_2023.shape, valid_2024.shape, full_train.shape, test_model.shape

((131400, 23), (131760, 23), (263160, 23), (131400, 23))

## 4. 모델 학습
기준선 선형회귀 모델(`LinearRegression`)과 트리 기반 모델(`LightGBM`, `CatBoost`)을 비교한다.

- `RMSE objective(제곱평균제곱근오차 기준 학습)`
- `Poisson objective(Poisson 분포 기준 학습)`

`Poisson objective`는 대여량처럼 count data(건수형 데이터)에 더 잘 맞을 수 있는지 확인하기 위한 비교 기준이다.


In [6]:
def evaluate_predictions(model_name: str, split_name: str, y_true: pd.Series, y_pred) -> dict:
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    return {
        'model': model_name,
        'split': split_name,
        'rmse': round(float(rmse), 4),
        'mae': round(float(mae), 4),
        'r2': round(float(r2), 4),
    }


results = []

lightgbm_rmse = LGBMRegressor(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    objective='regression',
)
lightgbm_rmse.fit(X_train, y_train, categorical_feature=CATEGORICAL_COLUMNS)
results.append(evaluate_predictions('LightGBM_RMSE', 'validation_2024', y_valid, lightgbm_rmse.predict(X_valid)))
lightgbm_rmse.fit(X_full, y_full, categorical_feature=CATEGORICAL_COLUMNS)
results.append(evaluate_predictions('LightGBM_RMSE', 'test_2025_refit', y_test, lightgbm_rmse.predict(X_test)))

lightgbm_poisson = LGBMRegressor(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    objective='poisson',
)
lightgbm_poisson.fit(X_train, y_train, categorical_feature=CATEGORICAL_COLUMNS)
results.append(evaluate_predictions('LightGBM_Poisson', 'validation_2024', y_valid, lightgbm_poisson.predict(X_valid)))
lightgbm_poisson.fit(X_full, y_full, categorical_feature=CATEGORICAL_COLUMNS)
results.append(evaluate_predictions('LightGBM_Poisson', 'test_2025_refit', y_test, lightgbm_poisson.predict(X_test)))

catboost_rmse = CatBoostRegressor(
    loss_function='RMSE',
    iterations=300,
    depth=8,
    learning_rate=0.05,
    random_seed=42,
    verbose=False,
)
catboost_rmse.fit(X_train, y_train, cat_features=CATEGORICAL_COLUMNS)
results.append(evaluate_predictions('CatBoost_RMSE', 'validation_2024', y_valid, catboost_rmse.predict(X_valid)))
catboost_rmse.fit(X_full, y_full, cat_features=CATEGORICAL_COLUMNS)
results.append(evaluate_predictions('CatBoost_RMSE', 'test_2025_refit', y_test, catboost_rmse.predict(X_test)))

catboost_poisson = CatBoostRegressor(
    loss_function='Poisson',
    iterations=300,
    depth=8,
    learning_rate=0.05,
    random_seed=42,
    verbose=False,
)
catboost_poisson.fit(X_train, y_train, cat_features=CATEGORICAL_COLUMNS)
results.append(evaluate_predictions('CatBoost_Poisson', 'validation_2024', y_valid, catboost_poisson.predict(X_valid)))
catboost_poisson.fit(X_full, y_full, cat_features=CATEGORICAL_COLUMNS)
results.append(evaluate_predictions('CatBoost_Poisson', 'test_2025_refit', y_test, catboost_poisson.predict(X_test)))

results_df = pd.DataFrame(results)
results_df

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.005186 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1490
[LightGBM] [Info] Number of data points in the train set: 131400, number of used features: 18
[LightGBM] [Info] Start training from score 0.722405


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.023423 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1525
[LightGBM] [Info] Number of data points in the train set: 263160, number of used features: 18
[LightGBM] [Info] Start training from score 0.724514


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001935 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1490
[LightGBM] [Info] Number of data points in the train set: 131400, number of used features: 18
[LightGBM] [Info] Start training from score -0.325170


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.004137 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1525
[LightGBM] [Info] Number of data points in the train set: 263160, number of used features: 18
[LightGBM] [Info] Start training from score -0.322255


,model,split,rmse,mae,r2
0,LightGBM_RMSE,validation_2024,1.0066,0.6121,0.5703
1,LightGBM_RMSE,test_2025_refit,0.8926,0.5455,0.5608
2,LightGBM_Poisson,validation_2024,1.0003,0.6074,0.5757
3,LightGBM_Poisson,test_2025_refit,0.8966,0.5401,0.5569
4,CatBoost_RMSE,validation_2024,1.0088,0.6139,0.5685
5,CatBoost_RMSE,test_2025_refit,0.9005,0.5480,0.5530
6,CatBoost_Poisson,validation_2024,1.0081,0.6096,0.5691
7,CatBoost_Poisson,test_2025_refit,0.8971,0.5456,0.5564


## 5. 피처 중요도(feature importance) 확인
`LightGBM_RMSE(RMSE 기준 LightGBM)` 모델의 feature importance(피처 중요도)를 확인한다.


In [7]:
metrics_path = OUTPUT_DATA_DIR / 'ddri_station_hour_model_metrics.csv'
results_df.to_csv(metrics_path, index=False, encoding='utf-8-sig')

importance_df = pd.DataFrame(
    {
        'feature': FEATURE_COLUMNS,
        'importance': lightgbm_rmse.feature_importances_,
    }
).sort_values('importance', ascending=False).reset_index(drop=True)

feature_label_map = {
    'station_id': '대여소 ID(Station ID)',
    'station_group': '대표 그룹(Station Group)',
    'cluster': '군집(Cluster)',
    'mapped_dong_code': '행정동 코드(Mapped Dong Code)',
    'hour': '시간대(Hour)',
    'weekday': '요일(Weekday)',
    'month': '월(Month)',
    'holiday': '공휴일 여부(Holiday)',
    'temperature': '기온(Temperature)',
    'humidity': '습도(Humidity)',
    'precipitation': '강수량(Precipitation)',
    'wind_speed': '풍속(Wind Speed)',
    'lag_1h': '1시간 전 대여량(Lag 1h)',
    'lag_24h': '24시간 전 대여량(Lag 24h)',
    'lag_168h': '168시간 전 대여량(Lag 168h)',
    'rolling_mean_24h': '24시간 이동평균(Rolling Mean 24h)',
    'rolling_mean_168h': '168시간 이동평균(Rolling Mean 168h)',
    'rolling_std_24h': '24시간 이동표준편차(Rolling Std 24h)',
}
importance_df['feature_kr'] = importance_df['feature'].map(feature_label_map).fillna(importance_df['feature'])

importance_path = OUTPUT_DATA_DIR / 'ddri_station_hour_lightgbm_feature_importance.csv'
importance_df.to_csv(importance_path, index=False, encoding='utf-8-sig')

plt.figure(figsize=(10, 6))
sns.barplot(data=importance_df.head(12), x='importance', y='feature_kr', hue='feature_kr', legend=False, palette='Blues_r')
plt.title('대표 대여소 시간대(Hour) LightGBM 피처 중요도')
plt.xlabel('중요도(Importance)')
plt.ylabel('피처(Feature)')
plt.tight_layout()
feature_image_path = IMAGE_DIR / 'ddri_station_hour_lightgbm_feature_importance.png'
plt.savefig(feature_image_path, dpi=150)
plt.close()

metrics_path, importance_path, feature_image_path

(PosixPath('/Users/cheng80/Desktop/ddri_work/works/05_prediction_long/output/data/ddri_station_hour_model_metrics.csv'),
 PosixPath('/Users/cheng80/Desktop/ddri_work/works/05_prediction_long/output/data/ddri_station_hour_lightgbm_feature_importance.csv'),
 PosixPath('/Users/cheng80/Desktop/ddri_work/works/05_prediction_long/output/images/ddri_station_hour_lightgbm_feature_importance.png'))

## 6. 비교 해석
`Poisson objective(Poisson 분포 기준 학습)`와 `CatBoost`가 `LightGBM_RMSE(RMSE 기준 LightGBM)` 대비 실제로 이득이 있는지 검토한다.
